In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

# Define your device
dev = qml.device("default.mixed", wires=1)


@qml.qnode(dev)
def random_gate(p, q, r):
    """
    Applies a Pauli X, Pauli Y, Pauli Z or does nothing at random.

    Args:
        - p (float): probability of applying Pauli X.
        - q (float): probability of applying Pauli Y.
        - r (float): probability of applying Pauli Z.

    Returns:
        - (np.tensor(float)): Measurement probabilities in the computational basis.

    """
    # The initial state is |0>.
    # We analyze the outcome probabilities for each possible gate application:
    #
    # 1. If Pauli X acts (with probability p):
    #    |0> -> X|0> = |1>.
    #    Measurement probabilities: P(0)=0, P(1)=1.
    #
    # 2. If Pauli Y acts (with probability q):
    #    |0> -> Y|0> = i|1>.
    #    Measurement probabilities (after taking absolute square): P(0)=0, P(1)=1.
    #
    # 3. If Pauli Z acts (with probability r):
    #    |0> -> Z|0> = |0>.
    #    Measurement probabilities: P(0)=1, P(1)=0.
    #
    # 4. If Identity I acts (with probability 1-p-q-r):
    #    |0> -> I|0> = |0>.
    #    Measurement probabilities: P(0)=1, P(1)=0.
    #
    # To find the total probability of measuring 0 or 1, we sum the probabilities
    # weighted by the likelihood of each gate being applied:
    #
    # Total P(0) = p * P(0|X) + q * P(0|Y) + r * P(0|Z) + (1-p-q-r) * P(0|I)
    # Total P(0) = p * 0 + q * 0 + r * 1 + (1-p-q-r) * 1
    # Total P(0) = r + 1 - p - q - r
    # Total P(0) = 1 - p - q
    #
    # Total P(1) = p * P(1|X) + q * P(1|Y) + r * P(1|Z) + (1-p-q-r) * P(1|I)
    # Total P(1) = p * 1 + q * 1 + r * 0 + (1-p-q-r) * 0
    # Total P(1) = p + q

    prob_0 = 1 - p - q
    prob_1 = p + q

    return np.array([prob_0, prob_1])


# These functions are responsible for testing the solution.

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    outs = random_gate(*ins).tolist()
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, rtol=1e-4), "Not the correct probabilities"

